In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_parallel_corpus(en_path, es_path, max_lines=None):
    with open(en_path, 'r', encoding='utf-8') as en_file, open(es_path, 'r', encoding='utf-8') as es_file:
        en_lines = en_file.readlines()
        es_lines = es_file.readlines()

    if max_lines:
        en_lines = en_lines[:max_lines]
        es_lines = es_lines[:max_lines]

    pairs = [(en.strip(), es.strip()) for en, es in zip(en_lines, es_lines)]
    return pairs

pairs = load_parallel_corpus('/content/drive/MyDrive/projects/Spanish Translator/europarl-v7.es-en.en', '/content/drive/MyDrive/projects/Spanish Translator/europarl-v7.es-en.es', max_lines=10)
print(pairs[0:5])


[('Resumption of the session', 'Reanudación del período de sesiones'), ('I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.', 'Declaro reanudado el período de sesiones del Parlamento Europeo, interrumpido el viernes 17 de diciembre pasado, y reitero a Sus Señorías mi deseo de que hayan tenido unas buenas vacaciones.'), ("Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series of natural disasters that truly were dreadful.", 'Como todos han podido comprobar, el gran "efecto del año 2000" no se ha producido. En cambio, los ciudadanos de varios de nuestros países han sido víctimas de catástrofes naturales verdaderamente terribles.'), ('You have requested a debate on this subject in the course of the next few days, during this part-session.', 'Sus 

In [ ]:
import random
import re

def clean_sentence(s):
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

# Load data
full_pairs = load_parallel_corpus('/content/drive/MyDrive/projects/Spanish Translator/europarl-v7.es-en.en',
                                  '/content/drive/MyDrive/projects/Spanish Translator/europarl-v7.es-en.es',
                                  max_lines=20000)

# Clean + filter
clean_pairs = []
MAX_LEN = 120 

for en, es in full_pairs:
    en = clean_sentence(en)
    es = clean_sentence(es)

    if len(en) == 0 or len(es) == 0:
        continue
    if len(en) > MAX_LEN or len(es) > MAX_LEN:
        continue

    clean_pairs.append((es, en))  # src=ES, tgt=EN

print("Total clean pairs:", len(clean_pairs))

random.seed(42)
random.shuffle(clean_pairs)

# make train/val/test sets
N = len(clean_pairs)
train_frac = 0.9
val_frac = 0.05

train_end = int(N * train_frac)
val_end = int(N * (train_frac + val_frac))

train_pairs = clean_pairs[:train_end]
val_pairs   = clean_pairs[train_end:val_end]
test_pairs  = clean_pairs[val_end:]

# unzip
train_es = [p[0] for p in train_pairs]
train_en = [p[1] for p in train_pairs]

val_es = [p[0] for p in val_pairs]
val_en = [p[1] for p in val_pairs]

test_es = [p[0] for p in test_pairs]
test_en = [p[1] for p in test_pairs]

print("Train:", len(train_es))
print("Val:", len(val_es))
print("Test:", len(test_es))

for i in range(3):
    print("\nES:", train_es[i])
    print("EN:", train_en[i])


Total clean pairs: 7659
Train: 6893
Val: 383
Test: 383

ES: (EN) En muchas de las contribuciones a este debate me parece percibir cierto grado de frustración.
EN: I sense some degree of frustration in many of the contributions to this debate.

ES: Las agresiones ni siquiera quedaban registradas.
EN: Atrocities were not even being recorded.

ES: Sra. diputada, se trata de una exhortación y no de una pregunta.
EN: Mrs Avilés-Perea, that was an exhortation rather than a question.


In [ ]:
with open("all_en.txt", "w", encoding="utf-8") as f_en, open("all_es.txt", "w", encoding="utf-8") as f_es:
    for en, es in full_pairs:
        if en.strip() and es.strip(): 
            f_en.write(en.strip() + "\n")
            f_es.write(es.strip() + "\n")

print("Saved:", len(full_pairs), "sentence pairs")

Saved: 20000 sentence pairs


In [ ]:
import os, math, random, time
from pathlib import Path
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import sacrebleu
from typing import List

# Tokenizer
class SPTokenizer:
    def __init__(self, model_path: str):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)

        self.id_bos = int(self.sp.bos_id())
        self.id_eos = int(self.sp.eos_id())
        self.id_pad = int(self.sp.pad_id())

    def encode(self, text: str) -> List[int]:
        if not isinstance(text, str):
            raise TypeError(f"Tokenizer expected string, got {type(text)}: {text}")

        ids = self.sp.encode(text, out_type=int)
        ids = [self.id_bos] + ids + [self.id_eos]
        return ids

    def decode(self, ids: List[int]) -> str:
        ids = [i for i in ids if i not in (self.id_bos, self.id_eos, self.id_pad)]
        return self.sp.decode(ids)

    def vocab_size(self):
        return self.sp.get_piece_size()

    def piece_to_id(self, piece: str):
        return self.sp.piece_to_id(piece)


In [ ]:
# Dataset + collate
class Seq2SeqDataset(Dataset):
    def __init__(self, src_sentences: List[str], tgt_sentences: List[str],
                 src_tokenizer: SPTokenizer, tgt_tokenizer: SPTokenizer, max_len=128):
        assert len(src_sentences) == len(tgt_sentences)
        self.src = src_sentences
        self.tgt = tgt_sentences
        self.src_tok = src_tokenizer
        self.tgt_tok = tgt_tokenizer
        self.max_len = max_len
        self.src_pad = self.src_tok.id_pad if self.src_tok.id_pad is not None else 0
        self.tgt_pad = self.tgt_tok.id_pad if self.tgt_tok.id_pad is not None else 0

        if self.src_pad < 0:
            self.src_pad = 0
        if self.tgt_pad < 0:
            self.tgt_pad = 0

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_ids = self.src_tok.encode(self.src[idx])[:self.max_len]
        tgt_ids = self.tgt_tok.encode(self.tgt[idx])[:self.max_len]

        src_ids = [min(max(x, 0), self.src_tok.vocab_size()-1) for x in src_ids]
        tgt_ids = [min(max(x, 0), self.tgt_tok.vocab_size()-1) for x in tgt_ids]

        src_ids = [x if x >= 0 else self.src_pad for x in src_ids]
        tgt_ids = [x if x >= 0 else self.tgt_pad for x in tgt_ids]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


def collate_fn(batch, pad_idx): 
    device = batch[0][0].device 

    src_batch = [item[0] for item in batch]
    tgt_batch = [item[1] for item in batch]

    max_src = max(len(s) for s in src_batch)
    max_tgt = max(len(t) for t in tgt_batch)

    src_tensor = torch.full((len(batch), max_src), fill_value=pad_idx, dtype=torch.long)
    tgt_tensor = torch.full((len(batch), max_tgt), fill_value=pad_idx, dtype=torch.long)

    for i, (src_ids, tgt_ids) in enumerate(batch):
        src_tensor[i, :len(src_ids)] = src_ids
        tgt_tensor[i, :len(tgt_ids)] = tgt_ids

    return src_tensor.to(device), tgt_tensor.to(device)


In [ ]:
# Pytorch transformer
class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.1, pad_idx=0):
        super().__init__()
        self.model_type = "transformer"
        self.d_model = d_model
        self.pad_idx = pad_idx

        # Embeddings
        self.src_tok_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=self.pad_idx)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=self.pad_idx)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True
        )

        # Output layer
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):

        src = src.long()
        tgt = tgt.long()

        # create masks with single padding index
        src_mask, tgt_mask, src_pad_mask, tgt_pad_mask = create_transformer_masks(
            src, tgt, pad_idx=self.pad_idx
        )

        # embeddings + positional encoding
        src_emb = self.pos_encoder(self.src_tok_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_tok_emb(tgt) * math.sqrt(self.d_model))

        # transformer forward
        out = self.transformer(
            src_emb, tgt_emb,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
            tgt_mask=tgt_mask
        )

        logits = self.fc_out(out)  # (batch, tgt_len, vocab)
        return logits


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :].to(x.device)
        return self.dropout(x)


In [ ]:

# Mask utilities
def create_transformer_masks(src, tgt, pad_idx):
    # src: (batch, src_len)
    # tgt: (batch, tgt_len)
    device = src.device
    src_pad_mask = (src == pad_idx)
    tgt_pad_mask = (tgt == pad_idx)
    # subsequent mask for target (prevent attending to future tokens)
    tgt_len = tgt.size(1)
    # PyTorch transformer expects mask with shape (tgt_len, tgt_len)
    tgt_mask = torch.triu(torch.ones((tgt_len, tgt_len), device=device), diagonal=1).bool()
    return None, tgt_mask, src_pad_mask, tgt_pad_mask


In [ ]:
# Training + eval helper functions
def shift_right(tgt_batch, bos_id):
    # convert target [B, T] -> decoder_input [B, T] shifted right with BOS at start
    B, T = tgt_batch.size()
    dec_input = torch.full((B, T), fill_value=bos_id, dtype=torch.long, device=tgt_batch.device)
    dec_input[:, 1:] = tgt_batch[:, :-1]
    # keep pad where original pad
    return dec_input

def train_epoch(model, dataloader, optimizer, criterion, device, tgt_bos_id, scaler):
    model.train()
    total_loss = 0.0
    for src_batch, tgt_batch in dataloader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        optimizer.zero_grad()

        decoder_input = shift_right(tgt_batch, tgt_bos_id)  # feed shifted right tgt tokens

        # Forward pass in mixed precision
        with torch.cuda.amp.autocast():
            logits = model(src_batch, decoder_input)  # (B, T, V)
            V = logits.size(-1)
            logits_flat = logits.view(-1, V)
            tgt_flat = tgt_batch.view(-1)
            loss = criterion(logits_flat, tgt_flat)

        # Backward with gradient scaling
        scaler.scale(loss).backward()

        # Gradient clipping (unscale first)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Optimizer step
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device, tgt_bos_id):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src_batch, tgt_batch in dataloader:
            src_batch = src_batch.to(device)
            tgt_batch = tgt_batch.to(device)
            decoder_input = shift_right(tgt_batch, tgt_bos_id)
            logits = model(src_batch, decoder_input)
            V = logits.size(-1)
            logits_flat = logits.view(-1, V)
            tgt_flat = tgt_batch.view(-1)
            loss = criterion(logits_flat, tgt_flat)
            total_loss += loss.item()
    return total_loss / len(dataloader)


In [ ]:
# Greedy + Beam decode
@torch.no_grad()
def greedy_decode(model, src_tensor, src_tokenizer, tgt_tokenizer, max_len=100, device='cpu'):
    model.eval()
    src = src_tensor.to(device).unsqueeze(0) if src_tensor.dim()==1 else src_tensor.to(device)

    # Initialize special token IDs and batch size
    bos = tgt_tokenizer.id_bos if tgt_tokenizer.id_bos is not None else 1
    eos = tgt_tokenizer.id_eos if tgt_tokenizer.id_eos is not None else 2
    pad = tgt_tokenizer.id_pad if tgt_tokenizer.id_pad is not None else 0
    B = src.size(0)

    # Get the max valid index (Vocab size - 1)
    V = tgt_tokenizer.vocab_size()
    max_index = V - 1

    generated = torch.full((B, 1), fill_value=bos, dtype=torch.long, device=device)

    for step in range(max_len):
        logits = model(src, generated)
        next_token_logits = logits[:, -1, :] 
        next_tokens = next_token_logits.argmax(dim=-1, keepdim=True)

        next_tokens = torch.clamp(next_tokens, max=max_index)

        generated = torch.cat([generated, next_tokens], dim=1)

        if (next_tokens == eos).all():
            break

    out = []
    for i in range(B):
        ids = generated[i].tolist()
        out.append(tgt_tokenizer.decode(ids))

    return out

@torch.no_grad()
def beam_search_decode(model, src_tensor, src_tokenizer, tgt_tokenizer, beam_width=5, max_len=100, device='cpu'):
    model.eval()
    src = src_tensor.to(device).unsqueeze(0) if src_tensor.dim()==1 else src_tensor.to(device)
    bos = tgt_tokenizer.id_bos if tgt_tokenizer.id_bos is not None else 1
    eos = tgt_tokenizer.id_eos if tgt_tokenizer.id_eos is not None else 2
    pad = tgt_tokenizer.id_pad if tgt_tokenizer.id_pad is not None else 0
    vocab_size = tgt_tokenizer.vocab_size()

    beams = [([bos], 0.0)]
    for _ in range(max_len):
        new_beams = []
        for tokens, score in beams:
            cur_input = torch.tensor([tokens], dtype=torch.long, device=device)
            logits = model(src, cur_input)
            probs = torch.log_softmax(logits[:, -1, :], dim=-1).squeeze(0) 
            topk_logp, topk_idx = probs.topk(beam_width)
            for k in range(beam_width):
                nt = topk_idx[k].item()
                ns = score + topk_logp[k].item()
                new_beams.append((tokens + [nt], ns))
        new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        beams = new_beams
        if all(b[0][-1] == eos for b in beams):
            break
    best_tokens = beams[0][0]
    return tgt_tokenizer.decode(best_tokens)


In [ ]:
# Training loop
def run_training(src_sentences_train, tgt_sentences_train,
                 src_sentences_val, tgt_sentences_val,
                 sp_model_path, device=None):

    import torch

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Tokenizers
    sp = SPTokenizer(sp_model_path)
    src_tok, tgt_tok = sp, sp 

    try:
        pad_idx = src_tok.piece_to_id("<pad>")
        bos_idx = tgt_tok.piece_to_id("<s>")
        eos_idx = tgt_tok.piece_to_id("</s>")
    except AttributeError:
        pad_idx, bos_idx, eos_idx = 0, 1, 2

    pad_idx = src_tok.id_pad if src_tok.id_pad is not None else 0 # This should be 3
    pad_idx = src_tok.id_pad if src_tok.id_pad is not None else 0
    if pad_idx < 0:
       pad_idx = 0


    print("Special token IDs -- PAD:", pad_idx, "BOS:", bos_idx, "EOS:", eos_idx)
    print("Tokenizer vocab size -- SRC:", src_tok.vocab_size(), "TGT:", tgt_tok.vocab_size())

    # Dataset + Dataloader
    train_ds = Seq2SeqDataset(src_sentences_train, tgt_sentences_train, src_tok, tgt_tok)
    val_ds   = Seq2SeqDataset(src_sentences_val, tgt_sentences_val, src_tok, tgt_tok)
    BATCH = 64
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_idx))
    val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_idx))

    # Precheck  dataset IDs
    def check_dataset(ds, vocab_size, name="Dataset"):
        for i, (src_ids, tgt_ids) in enumerate(ds):
            if src_ids.max() >= vocab_size or src_ids.min() < 0:
                print(f"{name} -- source IDs out of bounds at index {i}: min={src_ids.min()} max={src_ids.max()}")
            if tgt_ids.max() >= vocab_size or tgt_ids.min() < 0:
                print(f"{name} -- target IDs out of bounds at index {i}: min={tgt_ids.min()} max={tgt_ids.max()}")
        print(f"{name} check complete: all IDs within [0, {vocab_size-1}]")

    check_dataset(train_ds, src_tok.vocab_size(), "Train SRC")
    check_dataset(train_ds, tgt_tok.vocab_size(), "Train TGT")
    check_dataset(val_ds, src_tok.vocab_size(), "Val SRC")
    check_dataset(val_ds, tgt_tok.vocab_size(), "Val TGT")

    # CPU checks for min/max IDs
    all_src_ids = torch.cat([s for s, _ in train_ds] + [s for s, _ in val_ds])
    all_tgt_ids = torch.cat([t for _, t in train_ds] + [t for _, t in val_ds])

    print("CPU check: SRC min/max:", all_src_ids.min().item(), all_src_ids.max().item())
    print("CPU check: TGT min/max:", all_tgt_ids.min().item(), all_tgt_ids.max().item())

    src_vocab_size = src_tok.vocab_size()
    tgt_vocab_size = tgt_tok.vocab_size()
    print("Safe SRC vocab size:", src_vocab_size, "Safe TGT vocab size:", tgt_vocab_size)

    # Model
    print("Initializing model on CPU first for safety...")
    model = TransformerSeq2Seq(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
        dim_feedforward=2048, dropout=0.05,
        pad_idx=pad_idx
    )

    # Move to GPU
    print("Moving model to device:", device)
    model = model.to(device)

    # Loss, optimizer, scheduler
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

    def lr_lambda(step):
        warmup = 8000
        step = max(step, 1)
        return (1.0 / math.sqrt(512)) * min(1.0/math.sqrt(step), step*(warmup**-1.5))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # batch check
    src_batch, tgt_batch = next(iter(train_loader))
    print("DEBUG: train batch shapes:", src_batch.shape, tgt_batch.shape)
    print("DEBUG: train batch max SRC ID:", src_batch.max().item())
    print("DEBUG: train batch max TGT ID:", tgt_batch.max().item())

    # Training loop
    N_EPOCHS = 40
    best_val = float('inf')
    for epoch in range(1, N_EPOCHS + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, bos_idx)
        scheduler.step()
        val_loss = evaluate(model, val_loader, criterion, device, bos_idx)
        t1 = time.time()
        print(f"Epoch {epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f} time={t1-t0:.1f}s")

        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'epoch': epoch
            }, "best_transformer.pt")
            DRIVE_PATH = '/content/drive/MyDrive/projects/Spanish Translator/'

            torch.save({
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'epoch': epoch
            }, DRIVE_PATH + "best_transformer.pt")

    return model, src_tok, tgt_tok


In [ ]:
!spm_train \
  --input=all_es.txt,all_en.txt \
  --model_prefix=spm_shared \
  --vocab_size=16000 \
  --model_type=bpe \
  --character_coverage=1.0 \
  --input_sentence_size=1000000 \
  --shuffle_input_sentence=true \
  --user_defined_symbols="<pad>,<s>,</s>"



In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, src_tok, tgt_tok = run_training(
    src_sentences_train=train_es,
    tgt_sentences_train=train_en,
    src_sentences_val=val_es,
    tgt_sentences_val=val_en,
    sp_model_path="spm_shared.model",
    device=device
)

Special token IDs -- PAD: 0 BOS: 1 EOS: 2
Tokenizer vocab size -- SRC: 16000 TGT: 16000
Train SRC check complete: all IDs within [0, 15999]
Train TGT check complete: all IDs within [0, 15999]
Val SRC check complete: all IDs within [0, 15999]
Val TGT check complete: all IDs within [0, 15999]
CPU check: SRC min/max: 0 15999
CPU check: TGT min/max: 0 15993
Safe SRC vocab size: 16000 Safe TGT vocab size: 16000
Initializing model on CPU first for safety...
Moving model to device: cuda
DEBUG: train batch shapes: torch.Size([64, 28]) torch.Size([64, 27])
DEBUG: train batch max SRC ID: 15963
DEBUG: train batch max TGT ID: 15988
Epoch 1 train_loss=8.8908 val_loss=8.4694 time=31.1s
Epoch 2 train_loss=8.3782 val_loss=8.1865 time=32.9s
Epoch 3 train_loss=7.9889 val_loss=7.7112 time=35.6s
Epoch 4 train_loss=7.4770 val_loss=7.1805 time=35.0s
Epoch 5 train_loss=6.9444 val_loss=6.7133 time=34.8s
Epoch 6 train_loss=6.5156 val_loss=6.3801 time=35.3s
Epoch 7 train_loss=6.2230 val_loss=6.1694 time=34.0s
E

In [ ]:
import torch
import torch.nn as nn
import math
import sentencepiece as spm
import os
from typing import List

# Checkpoint load


class SPTokenizer:
    """Handles tokenization using a SentencePiece model."""
    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)
        self.id_pad = 0
        self.id_bos = 1
        self.id_eos = 2

    def vocab_size(self):
        return self.sp.get_piece_size()

    def encode(self, text):
        return [self.id_bos] + self.sp.encode_as_ids(text) + [self.id_eos]

    def decode(self, ids):
        clean_ids = [i for i in ids if i not in [self.id_bos, self.id_eos, self.id_pad]]
        return self.sp.decode_ids(clean_ids)


class PositionalEncoding(nn.Module):
    """Adds positional information to token embeddings."""
    def __init__(self, d_model, dropout=0.05, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerSeq2Seq(nn.Module):
    """The main Transformer model wrapper."""
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout, pad_idx):
        super().__init__()
        self.d_model = d_model
        self.src_tok_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)

        self.pos_encoder = PositionalEncoding(d_model, dropout)

        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.pad_idx = pad_idx

    def forward(self, src, tgt):
        src_mask = (src == self.pad_idx).to(src.device)
        tgt_mask = (tgt == self.pad_idx).to(tgt.device)
        tgt_size = tgt.size(1)
        subsequent_mask = self.transformer.generate_square_subsequent_mask(tgt_size).to(src.device)

        src_emb = self.pos_encoder(self.src_tok_emb(src) * self.d_model ** 0.5)
        tgt_emb = self.pos_encoder(self.tgt_tok_emb(tgt) * self.d_model ** 0.5)

        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_mask)
        output = self.transformer.decoder(tgt_emb, memory,
                                          tgt_mask=subsequent_mask,
                                          tgt_key_padding_mask=tgt_mask,
                                          memory_key_padding_mask=src_mask)

        return self.fc_out(output)

# model load
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pad_idx = 0
d_model = 512
TOKENIZER_PATH = 'spm_shared.model'

try:
    src_tok = SPTokenizer(TOKENIZER_PATH)
    tgt_tok = SPTokenizer(TOKENIZER_PATH)
    if not os.path.exists(TOKENIZER_PATH):
        print(f"Warning: Tokenizer file '{TOKENIZER_PATH}' not found. Ensure it's in the correct path.")
    print(f"Tokenizers loaded. Vocab Size: {src_tok.vocab_size()}")
except Exception as e:
    print(f"ERROR: Could not load tokenizer. Ensure '{TOKENIZER_PATH}' is in the correct directory. Details: {e}")

try:
    model = TransformerSeq2Seq(
        src_vocab_size=src_tok.vocab_size() if 'src_tok' in locals() else 32000,
        tgt_vocab_size=tgt_tok.vocab_size() if 'tgt_tok' in locals() else 32000,
        d_model=d_model,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.05,
        pad_idx=pad_idx
    ).to(device)

    # Load the checkpoint
    CHECKPOINT_PATH = "/content/drive/MyDrive/projects/Spanish Translator/best_transformer.pt"

    if not os.path.exists(CHECKPOINT_PATH):
        print(f"\n❌ ERROR: Checkpoint file not found at: {CHECKPOINT_PATH}. Did you mount Google Drive ('/content/drive')?")
    else:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(checkpoint["model_state"])

        model.eval()

        print(f"\n✅ Model successfully loaded from {CHECKPOINT_PATH} and set to evaluation mode on {device}.")
except Exception as e:
    print(f"\n❌ CRITICAL ERROR loading model architecture or weights: {e}")

✅ Tokenizers loaded. Vocab Size: 16000

✅ Model successfully loaded from /content/drive/MyDrive/projects/Spanish Translator/best_transformer.pt and set to evaluation mode on cuda.


In [ ]:
import torch

# Define a few custom Spanish sentences for testing
test_sentences_es = [
    "Ella se fue a la tienda por la mañana.",
    "¿Podrías ayudarme con este ejercicio de PyTorch?",
    "Me encanta programar en Python.",
    "El sol es caliente y el cielo es azul."
]

print("--- Quick Translation Check (Using Beam Search) ---")
print("-" * 60)

if 'model' not in globals() or 'src_tok' not in globals() or 'tgt_tok' not in globals():
    raise NameError("Model or Tokenizers (model, src_tok, tgt_tok) are not defined. Please run the model loading cell first.")
if 'beam_search_decode' not in globals():
    raise NameError("The 'beam_search_decode' function is not defined. Please run the cell containing the decode functions.")

model.eval()

# Translation Loop
with torch.no_grad():
    for i, src in enumerate(test_sentences_es):
        try:
            # Encode source
            src_ids = torch.tensor(src_tok.encode(src), dtype=torch.long)

            # Decode prediction
            pred = beam_search_decode(model, src_ids, src_tok, tgt_tok, beam_width=5, device=device)

            print(f"Source (ES): {src}")
            print(f"Prediction (EN): {pred}")
            print("-" * 60)

        except Exception as e:
            print(f"\n❌ ERROR during decoding sample: {src}. Details: {e}")
            break

--- Quick Translation Check (Using Beam Search) ---
------------------------------------------------------------
Source (ES): Ella se fue a la tienda por la mañana.
Prediction (EN): The first concerns the European Union has been taken by Mrs Palacio. The common position. It. The common position. The common position. The common position. It. The first. It. It. It. It. It. It is. The
------------------------------------------------------------
Source (ES): ¿Podrías ayudarme con este ejercicio de PyTorch?
Prediction (EN): Is there a Member who wishes to speak on this subject? policy?????????????????????????????????????
------------------------------------------------------------
Source (ES): Me encanta programar en Python.
Prediction (EN): I would like to congratulate you on the White Paper. It has been taken place in the process.m directive.m directive. There. It. It was in the future. It. It.m.m.m.m. It was
------------------------------------------------------------
Source (ES): El sol

In [ ]:
refs = []
hyps = []

for i in range(200):
    src = test_es[i]
    tgt = test_en[i]
    src_ids = torch.tensor(src_tok.encode(src), dtype=torch.long)
    pred = greedy_decode(model, src_ids, src_tok, tgt_tok, device=device)[0]

    refs.append([tgt])
    hyps.append(pred)

bleu = sacrebleu.corpus_bleu(hyps, refs)
print("BLEU:", bleu.score)